# DeepDream (codice su GitHub)

Il codice vive in `dreamlib.py` nel repository: lo modifichi su GitHub, qui lo riscarichi e lo usi.

**Ogni volta**
1. *Runtime → Cambia tipo di runtime → GPU* (solo la prima volta della sessione).
2. Esegui **Setup** (clona o aggiorna il repository).
3. Modifica i campi in **Parametri** ed eseguila, poi esegui **Genera**.

Se il repository e' privato, aggiungi il segreto `GITHUB_TOKEN` dall'icona della chiave a sinistra.

In [ ]:
#@title Setup (clone / aggiornamento da GitHub)
import os, sys, subprocess, importlib
import torch
import torch.nn.functional as F
from torchvision.transforms.functional import to_pil_image
from IPython.display import display, Image as IPyImage

REPO = "GiuliaBrix/DeepDream-lab"
CARTELLA = "DeepDream-lab"

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None

url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"


def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if token:
        out = out.replace(token, "***")
    return r.returncode, out


if not os.path.isdir(CARTELLA):
    rc, out = git("clone", url)
else:
    rc, out = git("-C", CARTELLA, "pull")
print(out if out else "ok")
if rc != 0:
    print("Problema con git: se il repository e' privato serve il segreto GITHUB_TOKEN (icona chiave a sinistra).")

if CARTELLA not in sys.path:
    sys.path.insert(0, CARTELLA)
import dreamlib
importlib.reload(dreamlib)
from dreamlib import Dreamer, resolve_layer, dream_octaves, make_start, device

d = None   # forza il ricaricamento della rete dopo ogni setup
print("Device:", device)

In [ ]:
#@title Parametri
modello = "vgg19" #@param ["vgg19", "vgg16", "googlenet", "resnet50", "alexnet"]
strato = "auto" #@param {type:"string"}
canale = -1 #@param {type:"integer"}
partenza = "rumore" #@param ["rumore", "immagine"]
immagine = "foto.jpg" #@param {type:"string"}
simmetria = "rotazionale" #@param ["nessuna", "speculare", "rotazionale", "entrambe"]
dimensione = 512 #@param {type:"integer"}
ottave = 4 #@param {type:"integer"}
scala_ottave = 1.4 #@param {type:"number"}
passi = 20 #@param {type:"integer"}
lr = 0.01 #@param {type:"number"}
jitter = 8 #@param {type:"integer"}

# strato = "auto" sceglie uno strato intermedio adatto al modello.
# canale = -1 significa "tutto lo strato".
# Per vedere gli strati di una rete: d.list_layers() (dopo aver eseguito Genera).

In [ ]:
#@title Genera
if d is None or d.name != modello:
    d = Dreamer(modello)

strato_usato = resolve_layer(d, modello, strato)
print("Modello:", modello, "| Rete caricata:", d.name, "| Strato:", strato_usato, "| Canale:", canale)
d.set_layer(strato_usato, None if canale < 0 else canale)

base = make_start(partenza, dimensione, immagine, square=(simmetria != "nessuna"))
result = dream_octaves(d, base, ottave, scala_ottave, passi, lr, jitter, simmetria)

out = to_pil_image(result.cpu()[0].clamp(0, 1))
out.save("dream.png")
display(out)

In [ ]:
#@title Zoom ricorsivo (effetto frattale) - parte dall'ultimo risultato
n_frame = 20 #@param {type:"integer"}
zoom = 1.05 #@param {type:"number"}
passi_per_frame = 10 #@param {type:"integer"}

x = result.clone()
H, W = x.shape[2:]
frames = [to_pil_image(x.cpu()[0].clamp(0, 1))]

for i in range(n_frame):
    h, w = int(H / zoom), int(W / zoom)
    top, left = (H - h) // 2, (W - w) // 2
    crop = x[:, :, top:top + h, left:left + w]
    x = F.interpolate(crop, size=(H, W), mode="bilinear", align_corners=False)
    x = dream_octaves(d, x, 2, scala_ottave, passi_per_frame, lr, jitter, simmetria)
    frames.append(to_pil_image(x.cpu()[0].clamp(0, 1)))

frames[0].save("zoom.gif", save_all=True, append_images=frames[1:], duration=120, loop=0)
display(IPyImage("zoom.gif"))

In [ ]:
# Elenco degli strati della rete caricata (per scegliere il parametro "strato")
d.list_layers()